# DAG Parameterization
The goal of this notebook is to assess which of the graphs (DAGs or PAGs) we generated during our causal structure learning step produces the best predictive model.

In [15]:
# Data manipulation and plotting
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re
import glob

#iterative imputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression


# Bayesian Networks
from pgmpy.base import DAG
from pgmpy.models import LinearGaussianBayesianNetwork, BayesianNetwork
from pgmpy.inference import VariableElimination

# Metrics
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, balanced_accuracy_score, f1_score
from sklearn.calibration import calibration_curve
from pycalib.metrics import ECE



import xgboost
import shap

## Data preparation

In [9]:
# Load training data
X_train = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_imp.csv', index_col=0)
synthetic_patients = pd.read_csv('../../Data Pre-processing/synthetic_patients_catch22.csv', index_col=0)
synth_outcome = synthetic_patients.pop('Outcome').values

print(X_train.shape,synthetic_patients.shape)

#Remove features with 0 variance
X_train = X_train.loc[:, X_train.var() >= 0.01]
synthetic_patients = synthetic_patients.filter(items=X_train.columns) # Keep only columns in X_train
print(X_train.shape, synthetic_patients.shape)

# # Impute missing values using Iterative Imputer (Linear Gaussian BN can't handle missing values)
# imputer = IterativeImputer(random_state=42, max_iter=100)
# X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
# X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train_imp = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_mice.csv', index_col=0)

y_train = pd.read_csv('../../Data Pre-processing/y_train_c12_w48_imp.csv', index_col=[0,1]).groupby('uid').max()


# Correlation feature selection
correlation_threshold = 0.3
X_train_corr = X_train_imp.loc[:, X_train_imp.corrwith(y_train['Outcome']).abs() >= correlation_threshold]
# synthetic_patients = synthetic_patients.filter(items=X_train_corr.columns) # Keep only columns in X_train_corr

print(X_train_imp.shape, synthetic_patients.shape)

(13054, 990) (100, 990)
(13054, 811) (100, 811)
(13054, 811) (100, 811)


# Loading DAGs

In [10]:
#Get all adjacency files
adjacency_files = glob.glob("../DAGs/*_adjacency.csv")

dags = {}
dags['Control'] = nx.from_edgelist([(col, 'Outcome') for col in X_train_imp.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
dags['Correlation'] = nx.from_edgelist([(col, 'Outcome') for col in X_train_corr.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
for file in adjacency_files:
    dag_name = re.search(r'../DAGs/(.*)_adjacency.csv', file).group(1)
    dag_name = dag_name.replace('x', '$\\cap$')
    dag_name = dag_name.replace(' + ', ' $\\cup$ ')
    dags[dag_name] = nx.DiGraph(pd.read_csv(file, index_col=0))

dags.pop('Golem $\\cap$ PCMB')  # Remove problematic DAG (has 0 nodes associated with Outcome)
list(dags.keys())

['Control',
 'Correlation',
 'Clinician Consensus $\\cup$ Golem',
 'Simplified Clinician Consensus $\\cup$ Simplified PCMB',
 'Simplified Clinician Consensus $\\cup$ Simplified Golem',
 'Clinician Consensus $\\cup$ PCMB',
 'Simplified Golem $\\cup$ Simplified PCMB',
 'Clinician Consensus',
 'Simplified Clinician Consensus',
 'Clinician Consensus $\\cap$ PCMB',
 'Golem',
 'PCMB',
 'Simplified Clinician Consensus $\\cap$ Simplified PCMB',
 'Simplified PCMB',
 'Clinician Consensus $\\cap$ Golem',
 'Simplified Golem',
 'Golem $\\cup$ PCMB',
 'Simplified Golem $\\cap$ Simplified PCMB',
 'Simplified Clinician Consensus $\\cap$ Simplified Golem']

# Model Training Functions
For each dag we train a Linear Gaussian Bayesian Network, an XGBoost

In [ ]:
dag = dags['Simplified Clinician Consensus $\\cup$ Simplified Golem']
nodes_in_dag = list(dag.nodes())
print(f"Number of nodes in DAG before removal: {len(nodes_in_dag)}")

# Remove drugs and interventions from DAG nodes
drugs = ['Midazolam', 'Fentanyl', 'Olanzapine', 'Haloperidol', 
                'Dexmedetomidine', 'Lorazepam', 'Morphine', 'Hydromorphone', 
                'Dopamine', 'Cisatracurium', 'Epinephrine', 'Norepinephrine', 
                'Milrinone', 'Dobutamine']
intervention_nodes = ['CRRT Therapy Type', 'ECMO Type', 'Ventilated', 'Pupillary Reaction', 'Peds Coma Score', 'Endotracheal tube', 'Weight']


nodes_in_dag = [node for node in nodes_in_dag if node not in drugs and node not in intervention_nodes]
print(f"Number of nodes in DAG after removal: {len(nodes_in_dag)}")

features_in_dag = [col for col in X_train_imp.columns if any(re.search(rf'^{node}(_.+)?$', col) for node in nodes_in_dag)]



# features_in_dag = X_train_imp.filter(items=nodes_in_dag).columns.tolist()
print(f"Number of nodes in DAG: {len(nodes_in_dag)}")
print(f"Number of features in DAG: {len(features_in_dag)}")



model = xgboost.XGBClassifier(objective='binary:logistic', random_state=42)
model.fit(X_train_imp.filter(features_in_dag), y_train.Outcome)

model.predict_proba(synthetic_patients.filter(features_in_dag))[:, 1]

# Check performance of model on synthetic patients
y_pred_proba = model.predict_proba(synthetic_patients.filter(features_in_dag))
auc_pr = average_precision_score(synth_outcome, y_pred_proba[:, 1])
auc_roc = roc_auc_score(synth_outcome, y_pred_proba[:, 1])
brier = brier_score_loss(synth_outcome, y_pred_proba[:, 1])
balanced_acc = balanced_accuracy_score(synth_outcome, model.predict(synthetic_patients.filter(features_in_dag)))
f1 = f1_score(synth_outcome, model.predict(synthetic_patients.filter(features_in_dag)))
print(f"AUC-PR: {auc_pr:.4f}")
print(f"AUC-ROC: {auc_roc:.4f}")
print(f"Brier Score: {brier:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"F1 Score: {f1:.4f}")


from sklearn.metrics import average_precision_score, precision_recall_curve
import numpy as np

# Model evaluation — use AUPRC
auprc = average_precision_score(synth_outcome, y_pred_proba[:, 1])

# Deployment — pick an operating threshold
precision, recall, thresholds = precision_recall_curve(synth_outcome, y_pred_proba[:, 1])

f1 = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-8)

optimal_threshold = thresholds[np.argmax(f1)]
print(f"Optimal Threshold: {optimal_threshold:.4f}")


# Confusion matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix
y_pred = model.predict(synthetic_patients.filter(features_in_dag))
cm = confusion_matrix(synth_outcome, y_pred_proba[:, 1] >= optimal_threshold)

#Fancy structured print of confusion matrix
print("Confusion Matrix:")
print(f" True Negatives: {cm[0, 0]} | False Positives: {cm[0, 1]} \n False Negatives: {cm[1, 0]} | True Positives: {cm[1, 1]}")

cm = confusion_matrix(synth_outcome, y_pred_proba[:, 1] >= 0.5)

#Fancy structured print of confusion matrix
print("Confusion Matrix:")
print(f" True Negatives: {cm[0, 0]} | False Positives: {cm[0, 1]} \n False Negatives: {cm[1, 0]} | True Positives: {cm[1, 1]}")

# SHAP feature importance
# Use the modern Explanation API so we get base_values and can verify additivity.
# model_output='raw' → log-odds space (correct for additive SHAP decomposition).
X_explain = synthetic_patients.filter(features_in_dag)
explainer = shap.TreeExplainer(model)
explanation = explainer(X_explain)

# Sanity check: base_value + sum(shap_values) should ≈ raw model output (log-odds)
raw_preds = model.predict(X_explain, output_margin=True)  # log-odds
shap_sum  = explanation.values.sum(axis=1) + explanation.base_values
max_err   = np.abs(raw_preds - shap_sum).max()
print(f'SHAP additivity check — max |raw_pred - (base + sum(shap))|: {max_err:.6f}')
if max_err > 0.01:
    print('WARNING: SHAP values do not sum to model output — something is wrong.')

values = pd.DataFrame(explanation.values, columns=X_explain.columns, index=X_explain.index)
values   

Number of nodes in DAG before removal: 23
Number of nodes in DAG after removal: 11
Number of nodes in DAG: 11
Number of features in DAG: 185
AUC-PR: 0.7681
AUC-ROC: 0.7320
Brier Score: 0.2723
Balanced Accuracy: 0.6100
F1 Score: 0.6723
Optimal Threshold: 0.7642
Confusion Matrix:
 True Negatives: 35 | False Positives: 15 
 False Negatives: 15 | True Positives: 35
Confusion Matrix:
 True Negatives: 21 | False Positives: 29 
 False Negatives: 10 | True Positives: 40
SHAP additivity check — max |raw_pred - (base + sum(shap))|: 0.000005


,Blood urea nitrogen__DN_HistogramMode_5,Blood urea nitrogen__DN_HistogramMode_10,Blood urea nitrogen__SB_BinaryStats_diff_longstretch0,Blood urea nitrogen__CO_f1ecac,Blood urea nitrogen__CO_FirstMin_ac,Blood urea nitrogen__SP_Summaries_welch_rect_area_5_1,Blood urea nitrogen__SP_Summaries_welch_rect_centroid,Blood urea nitrogen__FC_LocalSimple_mean3_stderr,Blood urea nitrogen__CO_trev_1_num,Blood urea nitrogen__IN_AutoMutualInfoStats_40_gaussian_fmmi,...,SpO2__IN_AutoMutualInfoStats_40_gaussian_fmmi,SpO2__MD_hrv_classic_pnn40,SpO2__SB_BinaryStats_mean_longstretch1,SpO2__FC_LocalSimple_mean1_tauresrat,SpO2__CO_Embed2_Dist_tau_d_expfit_meandiff,SpO2__SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1,SpO2__SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1,SpO2__PD_PeriodicityWang_th0_01,SpO2__DN_Mean,SpO2__DN_Spread_Std
uid,,,,,,,,,,,,,,,,,,,,,
syn_0_0,0.030223,0.229081,1.552888,0.0,-0.005742,-0.004747,-0.110935,-0.011068,0.046278,0.004018,...,0.015155,-0.004338,0.101632,0.035086,0.064619,-0.146242,-0.004893,0.105708,-0.015375,0.080436
syn_0_1,0.005361,-0.020293,3.249996,0.0,-0.041501,-0.039761,-0.141983,-0.006347,0.036761,-0.046718,...,0.000221,-0.243090,0.223697,0.042490,0.084070,-0.202326,-0.002988,-0.017745,-0.331549,0.008634
syn_0_2,0.006509,0.066784,4.273331,0.0,-0.024416,-0.038523,-0.123812,0.033019,0.036274,0.003468,...,0.001957,-0.109490,-0.053483,0.026462,-0.029275,0.097600,0.028688,-0.247953,-0.442416,-0.012369
syn_0_3,0.047844,-0.081765,4.501161,0.0,-0.004343,0.290062,-0.160894,0.081021,0.015040,0.015750,...,0.014233,-0.152667,-0.171747,-0.000308,-0.020370,0.123183,0.007665,-0.109317,-0.036591,0.074832
syn_0_4,0.012213,-0.053547,2.377168,0.0,-0.020648,0.053861,-0.203808,0.146782,-0.054365,0.012611,...,-0.021431,-0.078830,0.027062,0.064196,0.053251,0.098992,-0.003171,-0.087158,-0.289748,-0.131285
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
syn_1_45,-0.059459,-0.084603,2.095311,0.0,-0.004411,0.068889,-0.109653,-0.066585,-0.026589,0.004010,...,0.010037,-0.117877,-0.250522,0.030327,0.085086,-0.145639,-0.035247,0.024557,-0.124171,-0.051455
syn_1_46,0.005338,-0.051755,3.250101,0.0,-0.009652,0.059435,-0.295301,-0.033623,-0.024721,-0.050133,...,-0.029685,-0.234198,-0.183662,-0.022290,0.043692,-0.078730,-0.005918,0.024239,-0.376259,0.109552
syn_1_47,-0.005832,0.023665,3.924775,0.0,-0.008466,-0.037096,-0.179548,-0.072706,0.046440,0.051669,...,0.015849,0.095124,0.027341,0.035034,-0.005061,-0.081964,-0.008442,-0.223047,-0.203255,0.060219


In [26]:
model.save_model("/Users/eddie/Library/CloudStorage/OneDrive-UniversityofPittsburgh/Research/Projects/Dissertation/Aim 1/Aim 1.1 Causal Discovery/Predictive Modeling/trained_models/trained_model.json")

In [14]:
# Build an explicit column → DAG-node mapping using the same logic as feature selection.
# This is better than a naive suffix-strip because it anchors each feature to an actual
# node in the DAG, handles node names that contain underscores, and ignores any column
# that wasn't mapped to a node (there shouldn't be any, but this makes the assumption explicit).

col_to_node = {}
for col in values.columns:
    for node in nodes_in_dag:
        if re.search(rf'^{node}(_.+)?$', col):
            col_to_node[col] = node
            break  # each column belongs to exactly one node

unmapped = [c for c in values.columns if c not in col_to_node]
if unmapped:
    print(f"Warning: {len(unmapped)} columns had no matching DAG node and will be dropped: {unmapped[:5]}")

# Sum SHAP values across time windows for the same node, per patient.
# Summing (not averaging) is correct here: each window's SHAP value represents an
# additive contribution to log-odds, so the total contribution of a biomarker is their sum.
shap_by_node = (
    values[list(col_to_node)]          # drop any unmapped columns
    .rename(columns=col_to_node)
    .T.groupby(level=0).sum().T        # group duplicate column names, then sum
)

# Aggregate raw feature values the same way (mean across windows — for coloring in plots).
feat_by_node = (
    synthetic_patients.filter(list(col_to_node))
    .rename(columns=col_to_node)
    .T.groupby(level=0).mean().T
)

print(f"SHAP shape — before grouping : {values.shape}  (patients × time-windowed features)")
print(f"SHAP shape — after grouping  : {shap_by_node.shape}  (patients × DAG nodes)")
shap_by_node.head()


NameError: name 'values' is not defined

In [73]:
import seaborn as sns
from matplotlib.patches import Patch

dag_name = 'Simplified Clinician Consensus $\\cup$ Simplified Golem'
TOP_N = 15

# Rank nodes by mean |SHAP| — used to order all three plots consistently
mean_abs = shap_by_node.abs().mean().sort_values(ascending=False)
top_nodes = mean_abs.head(TOP_N).index.tolist()

# ── 1. Mean |SHAP| bar chart ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, len(top_nodes) * 0.45 + 1), dpi=150)
ax.barh(top_nodes[::-1], mean_abs[top_nodes[::-1]], color='steelblue', edgecolor='none')
ax.set_xlabel('Mean |SHAP value|', fontsize=12)
ax.set_title(f'Feature importance — {dag_name}', fontsize=11)
ax.grid(axis='x', alpha=0.35)
plt.tight_layout()
plt.savefig('../Predictive Modeling/Feature Importance/shap_bar.pdf', bbox_inches='tight', dpi=300)
plt.close()

# ── 2. Beeswarm (sign + magnitude + distribution) ─────────────────────────
# For each top node, jitter patients along the y-axis; colour by feature value
fig, ax = plt.subplots(figsize=(8, len(top_nodes) * 0.5 + 1.5), dpi=150)
rng = np.random.default_rng(42)

for i, node in enumerate(top_nodes[::-1]):
    sv   = shap_by_node[node].values
    fv   = feat_by_node[node].values if node in feat_by_node.columns else np.zeros_like(sv)
    # normalise feature value to [0,1] for colour mapping
    fv_n = (fv - np.nanmin(fv)) / (np.nanmax(fv) - np.nanmin(fv) + 1e-9)
    colours = plt.cm.coolwarm(fv_n)
    jitter  = rng.uniform(-0.35, 0.35, size=len(sv))
    ax.scatter(sv, i + jitter, c=colours, alpha=0.45, s=14, linewidths=0, zorder=3)

ax.set_yticks(range(len(top_nodes)))
ax.set_yticklabels(top_nodes[::-1], fontsize=10)
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('SHAP value (summed across time windows)', fontsize=12)
ax.set_title(f'SHAP beeswarm — {dag_name}', fontsize=11)
ax.grid(axis='x', alpha=0.3)

# Colourbar for feature value
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.01, fraction=0.025)
cbar.set_label('Feature value (low → high)', fontsize=9)
cbar.set_ticks([0, 1]); cbar.set_ticklabels(['Low', 'High'])

plt.tight_layout()
plt.savefig('../Predictive Modeling/Feature Importance/shap_beeswarm.pdf', bbox_inches='tight', dpi=300)
plt.close()

# ── 3. SHAP dependence plots for top 3 nodes ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=150)
for ax, node in zip(axes, top_nodes[:3]):
    sv = shap_by_node[node].values
    fv = feat_by_node[node].values if node in feat_by_node.columns else np.zeros_like(sv)
    sc = ax.scatter(fv, sv, c=sv, cmap='coolwarm', alpha=0.55, s=18,
                    norm=plt.Normalize(-np.abs(sv).max(), np.abs(sv).max()), linewidths=0)
    ax.axhline(0, color='black', lw=0.7, ls='--')
    ax.set_xlabel(node, fontsize=11)
    ax.set_ylabel('SHAP value', fontsize=11)
    ax.set_title(f'Dependence — {node}', fontsize=10)
    ax.grid(alpha=0.3)
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04).set_label('SHAP', fontsize=8)

plt.tight_layout()
plt.savefig('../Predictive Modeling/Feature Importance/shap_dependence.pdf', bbox_inches='tight', dpi=300)
plt.close()

In [13]:
# ── SHAP dependence plots for ALL biomarkers ──────────────────────────────
# x-axis clipped to 2nd–98th percentile so outliers don't compress the bulk of the data.
all_nodes = mean_abs.index.tolist()  # ordered by importance (descending)
n_nodes = len(all_nodes)
n_cols = 3
n_rows = int(np.ceil(n_nodes / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), dpi=150)
axes = axes.flatten()

for i, node in enumerate(all_nodes):
    ax = axes[i]
    sv = shap_by_node[node].values
    fv = feat_by_node[node].values if node in feat_by_node.columns else np.zeros_like(sv)

    # Clip x-axis to 2nd–98th percentile to show the actual variance
    lo, hi = np.percentile(fv, [2, 98])
    pad = (hi - lo) * 0.05 + 1e-9  # small buffer so edge points aren't cut
    xlim = (lo - pad, hi + pad)

    sc = ax.scatter(fv, sv, c=sv, cmap='coolwarm', alpha=0.6, s=20,
                    norm=plt.Normalize(-np.abs(sv).max(), np.abs(sv).max()),
                    linewidths=0, edgecolors='none')
    ax.axhline(0, color='black', lw=0.7, ls='--')
    ax.set_xlim(xlim)
    ax.set_xlabel(node, fontsize=10)
    ax.set_ylabel('SHAP value', fontsize=10)
    ax.set_title(node, fontsize=11, fontweight='bold')
    ax.grid(alpha=0.25)

# Hide unused subplots
for j in range(n_nodes, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('../Predictive Modeling/Feature Importance/shap_dependence_all.pdf',
            bbox_inches='tight', dpi=300)
plt.close()

NameError: name 'mean_abs' is not defined